# 1. Dataset Preparation

This notebook documents and inspects the saved Stage 1 outputs. Its code cell only reads existing CSV/JSON metadata; it does not decode images, regenerate metadata or splits, fit preprocessing statistics, or train models.

The local RSNA Stage 2 files are expected under `Data/`: `stage_2_train_images/`, `stage_2_train_labels.csv`, and `stage_2_detailed_class_info.csv`. The dataset is not included in this repository.

To build metadata from a local dataset, the existing command is `python scripts/create_metadata.py`. It maps `Lung Opacity` to Pneumonia (1), `Normal` to Healthy (0), excludes other classes, and retains multiple bounding boxes as aligned coordinate arrays. `python scripts/create_split.py` creates the seed-42 stratified 70/20/10 image-ID split. Do not run these commands when preserving an existing experiment split.

## Shared preprocessing

Images are decoded on demand, resized to 128 × 128 grayscale, scaled to [0, 1], and standardized using global mean and standard deviation fitted on training images only. The existing saved statistics are in `metadata/preprocessing_stats.json`; the fixed split is in `metadata/split_metadata.csv`.

The project scripts and shared implementation live in `scripts/create_metadata.py`, `scripts/create_split.py`, `src/data/dataset.py`, and `src/data/preprocessing.py`.

In [ ]:
import csv
import json
from pathlib import Path

ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "metadata").is_dir() and (candidate / "src").is_dir():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate the project root from the notebook directory")

metadata_path = ROOT / "metadata" / "pneumonia_metadata.csv"
split_path = ROOT / "metadata" / "split_metadata.csv"
stats_path = ROOT / "metadata" / "preprocessing_stats.json"
for path in (metadata_path, split_path, stats_path):
    if not path.is_file():
        raise FileNotFoundError(f"Required saved preprocessing output is missing: {path}")

with metadata_path.open(newline="", encoding="utf-8-sig") as stream:
    metadata_rows = list(csv.DictReader(stream))
with split_path.open(newline="", encoding="utf-8-sig") as stream:
    split_rows = list(csv.DictReader(stream))
stats = json.loads(stats_path.read_text(encoding="utf-8"))

print(f"Saved labeled metadata rows: {len(metadata_rows):,}")
for split in ("train", "validation", "test"):
    rows = [row for row in split_rows if row["split"] == split]
    positive = sum(row["label"] == "1" for row in rows)
    print(f"{split}: {len(rows):,} images ({positive:,} Pneumonia, {len(rows) - positive:,} Healthy)")
print(f"Resize: {stats['image_size'][0]} x {stats['image_size'][1]}")
print(f"Train-only mean/std: {stats['mean']:.10f} / {stats['std']:.10f}")
print(f"Normalization fit image count: {stats['fit_image_count']:,}")